In [1]:
import os
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [2]:
W2V_SIZE = 200                 #Розмір векторного простору (кількість ознак на слово)
W2V_WINDOW = 5                 #Розмір контекстного вікна
W2V_MIN_COUNT = 2              #Мінімальна кількість появ слова для врахування у словнику
W2V_EPOCHS = 10                #Кількість епох тренування Word2Vec
TEST_SIZE = 0.2                #Частка даних для тестування (20%)
RANDOM_STATE = 42           
PCA_COMPONENTS = [None, 50, 100, 200] #Кількість компонент для PCA (зменшення розмірності)
OUTPUT_DIR = "lab3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
#Завантаження датасету
df = pd.read_csv("Corona_NLP_train.csv", encoding='latin1')
print("Columns:", df.columns.tolist())

Columns: ['UserName', 'ScreenName', 'Location', 'TweetAt', 'OriginalTweet', 'Sentiment']


In [4]:
#Автоматичне визначення назв колонок для тексту та міток
text_col = None
label_col = None

#Перебір усіх назв стовпців, щоб знайти ті, які відповідають тексту або емоціям
for c in df.columns:
    if c.lower().strip() in ['original tweet','original_tweet','tweet','text','content']:
        text_col = c
    if c.lower().strip() in ['sentiment','label','label (sentiment)']:
        label_col = c

#Якщо колонки не знайдені за назвою — встановлюється за замовчуванням
if text_col is None:
    text_col = 'OriginalTweet' if 'OriginalTweet' in df.columns else df.columns[0]
if label_col is None:
    label_col = 'Sentiment' if 'Sentiment' in df.columns else df.columns[-1]

print(f"Using text column: {text_col}, label column: {label_col}")

Using text column: OriginalTweet, label column: Sentiment


In [5]:
#Уніфікація нейтральних міток — усі варіанти ('Neutral', 'neutral', 'NEUTRAL') замінюються на 'Other'
df[label_col] = df[label_col].replace({'Neutral':'Other','neutral':'Other','NEUTRAL':'Other'})

In [6]:
import nltk
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STOPWORDS = set(ENGLISH_STOP_WORDS)

def preprocess_text(text):
    #Якщо значення не є рядком — повертається порожній список
    if not isinstance(text, str):
        return []
    text = text.lower()
    #Видалення посилань, згадок, хештегів і небуквених символів
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    #Токенізація — розбиття тексту на окремі слова
    tokens = re.findall(r'\b[a-z0-9]+\b', text)
    #Видалення стоп-слів, цифр і дуже коротких токенів
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1 and not t.isdigit()]
    return tokens

#Застосування попередньої обробки до всіх твітів
print("Preprocessing texts (tokenization, stopword removal)...")
df['tokens'] = df[text_col].astype(str).apply(preprocess_text)
df['tokens_count'] = df['tokens'].apply(len)
print("Example tokens:")
print(df[['tokens','tokens_count']].head())

Preprocessing texts (tokenization, stopword removal)...
Example tokens:
                                              tokens  tokens_count
0                                                 []             0
1  [advice, talk, neighbours, family, exchange, p...            27
2  [coronavirus, australia, woolworths, elderly, ...            11
3  [food, stock, don, panic, food, need, stay, ca...            17
4  [ready, supermarket, covid19, outbreak, parano...            18


In [7]:
print("Training Word2Vec (this may take time)...")
sentences = df['tokens'].tolist()
w2v = Word2Vec(sentences=sentences, vector_size=W2V_SIZE, window=W2V_WINDOW, min_count=W2V_MIN_COUNT, workers=4, epochs=W2V_EPOCHS)
print("Word2Vec vocab size:", len(w2v.wv.index_to_key))

#Збереження навченої моделі для подальшого використання
w2v_path = os.path.join(OUTPUT_DIR, 'w2v.kv')
w2v.wv.save(w2v_path)
print("Saved Word2Vec keyed vectors to:", w2v_path)

Training Word2Vec (this may take time)...
Word2Vec vocab size: 20516
Saved Word2Vec keyed vectors to: lab3_outputs\w2v.kv


In [8]:
def tweet_vector(tokens, model, size):
    vec = np.zeros(size, dtype=float)
    count = 0
    for t in tokens:
        if t in model.wv:
            vec += model.wv[t]   #додаавання вектору кожного слова
            count += 1
    if count > 0:
        vec /= count             #усереднення векторів, щоб отримати один для твіту
    return vec

#Матриця ознак для всіх твітів
X = np.vstack(df['tokens'].apply(lambda toks: tweet_vector(toks, w2v, W2V_SIZE)).tolist())
print("Embeddings matrix shape:", X.shape)

Embeddings matrix shape: (41157, 200)


In [9]:
le = LabelEncoder()
y = le.fit_transform(df[label_col].astype(str))
labels = le.classes_
print("Classes:", labels)

Classes: ['Extremely Negative' 'Extremely Positive' 'Negative' 'Other' 'Positive']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
print("Train/test shapes:", X_train.shape, X_test.shape)

Train/test shapes: (32925, 200) (8232, 200)


In [12]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

#Словник моделей для порівняння
classifiers = {
'LogisticRegression': LogisticRegression(max_iter=1000),
'SVM': SVC(probability=False),
'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
'GaussianNB': GaussianNB()
}

In [13]:
results = []
for pca_n in PCA_COMPONENTS:
    for name, clf in classifiers.items():
        #Якщо PCA вимагає більше ознак, ніж є, пропускаємо
        if pca_n is not None and pca_n > X_train.shape[1]:
            print(f"Skipping PCA={pca_n} because feature dim={X_train.shape[1]}")
            continue

        #Побудова пайплайну: масштабування - PCA - модель
        steps = [('scaler', StandardScaler())]
        if pca_n is not None:
            steps.append(('pca', PCA(n_components=pca_n, random_state=RANDOM_STATE)))
        steps.append(('clf', clf))
        pipe = Pipeline(steps)
        
        print(f"Training {name} with PCA={pca_n}...")
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        #Оцінка результатів
        acc = accuracy_score(y_test, y_pred)
        prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
        report = classification_report(y_test, y_pred, target_names=labels, zero_division=0)
        cm = confusion_matrix(y_test, y_pred)

        #Збереження результатів для подальшого аналізу
        results.append({
            'model': name,
            'pca': 'None' if pca_n is None else pca_n,
            'accuracy': acc,
            'precision_macro': prec_macro,
            'recall_macro': rec_macro,
            'f1_macro': f1_macro,
            'classification_report': report,
            'confusion_matrix': cm
        })
        print(f"Done: {name}, PCA={pca_n}, acc={acc:.4f}, f1_macro={f1_macro:.4f}")

Training LogisticRegression with PCA=None...


C:\Users\vovaf\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Done: LogisticRegression, PCA=None, acc=0.4525, f1_macro=0.4512
Training SVM with PCA=None...
Done: SVM, PCA=None, acc=0.4401, f1_macro=0.4372
Training RandomForest with PCA=None...
Done: RandomForest, PCA=None, acc=0.4269, f1_macro=0.4227
Training GaussianNB with PCA=None...
Done: GaussianNB, PCA=None, acc=0.3096, f1_macro=0.3010
Training LogisticRegression with PCA=50...
Done: LogisticRegression, PCA=50, acc=0.3929, f1_macro=0.3891
Training SVM with PCA=50...
Done: SVM, PCA=50, acc=0.4312, f1_macro=0.4275
Training RandomForest with PCA=50...
Done: RandomForest, PCA=50, acc=0.4294, f1_macro=0.4217
Training GaussianNB with PCA=50...
Done: GaussianNB, PCA=50, acc=0.3280, f1_macro=0.3289
Training LogisticRegression with PCA=100...
Done: LogisticRegression, PCA=100, acc=0.4297, f1_macro=0.4287
Training SVM with PCA=100...
Done: SVM, PCA=100, acc=0.4399, f1_macro=0.4369
Training RandomForest with PCA=100...
Done: RandomForest, PCA=100, acc=0.4383, f1_macro=0.4269
Training GaussianNB with P

C:\Users\vovaf\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Done: LogisticRegression, PCA=200, acc=0.4524, f1_macro=0.4511
Training SVM with PCA=200...
Done: SVM, PCA=200, acc=0.4401, f1_macro=0.4372
Training RandomForest with PCA=200...
Done: RandomForest, PCA=200, acc=0.4294, f1_macro=0.4125
Training GaussianNB with PCA=200...
Done: GaussianNB, PCA=200, acc=0.3015, f1_macro=0.3016


In [14]:
summary = []
for r in results:
    summary.append({
        'Model': r['model'],
        'PCA': r['pca'],
        'Accuracy': r['accuracy'],
        'Precision_macro': r['precision_macro'],
        'Recall_macro': r['recall_macro'],
        'F1_macro': r['f1_macro']
    })

#Збереження результатів у CSV
summary_df = pd.DataFrame(summary)
summary_csv = os.path.join(OUTPUT_DIR, 'summary_metrics.csv')
summary_df.to_csv(summary_csv, index=False)
print("Saved summary to:", summary_csv)

Saved summary to: lab3_outputs\summary_metrics.csv


In [16]:
plt.figure(figsize=(8,6))
for model in summary_df['Model'].unique():
    sub = summary_df[summary_df['Model'] == model]
    x = sub['PCA'].replace({'None':0})
    x = x.astype(int)
    y = sub['Accuracy']
    plt.plot(x, y, marker='o', label=model)
plt.xlabel('PCA components (0 = None)')
plt.ylabel('Accuracy')
plt.title('PCA effect on accuracy for models')
plt.legend()
plot_path = os.path.join(OUTPUT_DIR, 'pca_accuracy.png')
plt.savefig(plot_path)
plt.close()
print('Saved PCA plot to:', plot_path)

C:\Users\vovaf\AppData\Local\Temp\ipykernel_17176\647648432.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x = sub['PCA'].replace({'None':0})
C:\Users\vovaf\AppData\Local\Temp\ipykernel_17176\647648432.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x = sub['PCA'].replace({'None':0})
C:\Users\vovaf\AppData\Local\Temp\ipykernel_17176\647648432.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=F

Saved PCA plot to: lab3_outputs\pca_accuracy.png


In [17]:
best = max(results, key=lambda rr: rr['f1_macro'])
print('Best model:', best['model'], 'PCA:', best['pca'], 'Accuracy:', best['accuracy'])
print(best['classification_report'])

Best model: LogisticRegression PCA: None Accuracy: 0.4525024295432459
                    precision    recall  f1-score   support

Extremely Negative       0.53      0.32      0.40      1096
Extremely Positive       0.58      0.39      0.46      1325
          Negative       0.40      0.43      0.41      1983
             Other       0.50      0.58      0.53      1543
          Positive       0.41      0.49      0.44      2285

          accuracy                           0.45      8232
         macro avg       0.48      0.44      0.45      8232
      weighted avg       0.46      0.45      0.45      8232



In [18]:
cm = best['confusion_matrix']
fig, ax = plt.subplots(figsize=(6,5))
ax.imshow(cm, interpolation='nearest')
ax.set_title(f"Confusion Matrix: {best['model']} (PCA={best['pca']})")
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45)
ax.set_yticklabels(labels)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='w')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'best_confusion_matrix.png')
plt.savefig(cm_path)
plt.close()
print('Saved confusion matrix to:', cm_path)

Saved confusion matrix to: lab3_outputs\best_confusion_matrix.png


In [19]:
with open(os.path.join(OUTPUT_DIR, 'detailed_results.txt'), 'w', encoding='utf-8') as f:
    for r in results:
        f.write(f"MODEL: {r['model']} | PCA: {r['pca']}\n")
        f.write(f"Accuracy: {r['accuracy']:.4f} | F1_macro: {r['f1_macro']:.4f}\n")
        f.write("Classification Report:\n")
        f.write(r['classification_report'] + "\n")
        f.write("Confusion Matrix:\n")
        f.write(str(r['confusion_matrix']) + "\n")
        f.write("="*80 + "\n\n")

print('All done. Outputs saved under', OUTPUT_DIR)

All done. Outputs saved under lab3_outputs


In [2]:
#Висновок: Після тренування всіх моделей (враховуючи додавання та зміну PCA до), найкращі результати вивела логістична регресія (без PCA і PCA=200,
#результати несуттєво відрізняються лише на 0.0001)

#Зміна PCA змінила якість класифікації:
#при PCA = 50, результати більшості моделей покращилися;
#при PCA = 100 - погіршилися;
#при PCA = 200 - результати повернулися до початкових, без використання PCA
#PCA не суттєво покращує точність моделей на Word2Vec. Можливо, інформація, яка важлива для тональності, 
#все одно розподілена по багатьом вимірам, і сильне зменшення лише шкодить класифікації.

#Якщо пояснювати далі результати, SVM справився трішки гірше за LR
#Random Forest, більш за все, важко працював з розрідженими або щільними Word2Vec-векторами, тож і результат виявися низьким
#GaussianNB - явно не підходить для таких даних — ймовірно, через порушення припущення про нормальний розподіл ознак.
#Лінійні моделі краще справляються з Word2Vec ембеддінгами, особливо логістична регресія.

#В найкращій моделі (LR, PCA = None) клас "Other" передається найкраще (F1≈0.53), тобто модель добре розрізняє нейтральні твіти
#Екстремальні класи (Extremely Negative/Positive) мають високий precision, але низький recall.
#Precision ≈ 0.53–0.58 - коли модель передбачає екстремальний твіт, вона часто права.
#Recall ≈ 0.32–0.39 - але вона пропускає більшість екстремальних твітів.
#Negative/Positive середні по точності та recall, F1≈0.41–0.44

#Модель більш консервативна щодо екстремальних класів — виявляє тільки явні випадки, але багато пропускає. Клас "Other" найстабільніший, 
#що типово для твітер-тональності, бо більшість твітів нейтральні або неявно виражені.